## Dream 2000 Phones Scrapping

In [1]:
import selenium 

In [ ]:

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import NoSuchElementException
import time

## -- Options --
options = Options()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)
options.add_argument("--window-size=1920,1080")
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")


## -- Phones List --
Phones = []


## -- Scraping --
driver = webdriver.Chrome(options=options)
driver.get("https://dream2000.com/en/collections/mobiles")
time.sleep(5)

page = 1

while True:

    page_products = 0

    ## -- Product Cards --
    cards = driver.find_elements(By.CSS_SELECTOR,'.product-card__wrapper.color-scheme-theme.anm-reveal-item.overflow-hidden')

    for card in cards:

        try:
            Product = card.find_element(By.CSS_SELECTOR, 'a[href*="/en/products/"]')

            Name = Product.get_attribute('aria-label')
            Link = Product.get_attribute('href')

            Image = card.find_element(By.CSS_SELECTOR, 'img.product-card__image').get_attribute('src')

        except NoSuchElementException:
            continue

        ## -- Price --
        try:
            Price = card.find_element(By.CSS_SELECTOR, '.price').text

        except NoSuchElementException:
            Price = "Product Not Available"

        ## -- Stock --
        try:
            Stock = card.find_element(By.CSS_SELECTOR,'[data-inventory-status]').get_attribute('data-inventory-status')

        except NoSuchElementException:
            Stock = "Product Not Available"

        Phones.append({
            'Name': Name,
            'Link': Link,
            'Image': Image,
            'Price': Price,
            'Stock': Stock
        })

        page_products += 1

    ## -- Page Summary --

    print(
        f"Page {page} finished successfully. "
        f"Products collected on this page: {page_products}. "
        f"Total products collected: {len(Phones)}"
    )

    ## -- Next Page --
    next_buttons = driver.find_elements(
        By.CSS_SELECTOR, "a[aria-label='Next']"
    )

    if not next_buttons:
        print("No Next page. Scraping finished.")
        break

    next_url = next_buttons[0].get_attribute('href')

    if not next_url or next_url == driver.current_url:
        print("Invalid or unchanged Next URL.")
        break

    print("Navigating to:", next_url)

    driver.get(next_url)

    page += 1
    time.sleep(5)


## -- Final Result --
print(f"\nScraping completed. Total products: {len(Phones)}")


In [ ]:
Phones

## Import To CSV & Parquet File

In [33]:
import pandas as pd

In [35]:
df_Phones = pd.DataFrame(Phones)

In [36]:
df_Phones.head()

,Name,Link,Image,Price,Stock
0,Samsung Galaxy A17,https://dream2000.com/en/products/galaxy-a17,https://dream2000.com/cdn/shop/files/2025-09-2...,"LE 19,990.00\nLE 21,000.00",in-stock
1,Samsung Galaxy A07,https://dream2000.com/en/products/galaxy-a07,https://dream2000.com/cdn/shop/files/2025-09-1...,"LE 13,490.00\nLE 14,200.00",low-stock
2,Xiaomi Redmi 15C,https://dream2000.com/en/products/redmi-15c,https://dream2000.com/cdn/shop/files/2025-09-0...,"LE 15,150.00\nLE 16,000.00",in-stock
3,Honor X9d 5G,https://dream2000.com/en/products/honor-x9d-5g...,https://dream2000.com/cdn/shop/files/2025-10-3...,"LE 26,990.00\nLE 28,400.00",in-stock
4,Xiaomi Poco C85,https://dream2000.com/en/products/poco-c85,https://dream2000.com/cdn/shop/files/2025-11-0...,"LE 13,990.00\nLE 14,700.00",in-stock


In [37]:
df_Phones.to_csv("Phones.csv" , index = False)
df_Phones.to_parquet("Phones.parquet" , index = False)